# WaterDropML: train the U-Net on Google Colab

Before you start: **Runtime → Change runtime type → T4 GPU**.

On your laptop, from the WaterDropML folder, build the bundle (PowerShell):
```
python src/unet/prepare_data.py
Compress-Archive -Path src, data\unet -DestinationPath unet_bundle.zip -Force
```
Then run the cells below in order. See `docs/UNET_GUIDE.md` for the full workflow.

## 1. Upload `unet_bundle.zip`

In [ ]:
from google.colab import files
up = files.upload()   # choose unet_bundle.zip
!rm -rf /content/wd && mkdir -p /content/wd && unzip -q unet_bundle.zip -d /content/wd
%cd /content/wd
!ls data/unet && head -3 data/unet/meta.csv

## 2. Install dependencies

In [ ]:
!pip -q install segmentation-models-pytorch opencv-python-headless
import torch; print('torch', torch.__version__, '| GPU:', torch.cuda.is_available())

## 3. Train

This takes about 10–20 min on a T4. Add `--gt-only` once `data/masks_gt/` has enough hand-corrected masks.

In [ ]:
!python src/unet/train.py --epochs 60 --batch 8 --crop 384

## 4. Look at the learning curves

Validation IoU should rise and level off, and the boundary error (full-res px) should fall.

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
log = pd.read_csv('models/train_log.csv')
fig, ax = plt.subplots(1, 3, figsize=(14, 3.5))
log.plot(x='epoch', y='train_loss', ax=ax[0], legend=False, title='train loss')
log.plot(x='epoch', y='val_iou', ax=ax[1], legend=False, title='val IoU')
log.plot(x='epoch', y='val_boundary_px_full', ax=ax[2], legend=False, title='val boundary error (px)')
plt.tight_layout(); log.tail()

## 5. Quick visual check on a validation image

In [ ]:
import sys, cv2, numpy as np
sys.path.insert(0, 'src/unet')
from common import build_model, predict_prob, pick_device
dev = pick_device(); ck = torch.load('models/unet_best.pt', map_location=dev, weights_only=False)
m = build_model(ck['encoder'], pretrained=False); m.load_state_dict(ck['state_dict']); m.to(dev)
meta = pd.read_csv('data/unet/meta.csv'); stem = meta[meta.split=='val'].stem.iloc[0]
img = cv2.imread(f'data/unet/images/{stem}.png'); gt = cv2.imread(f'data/unet/masks/{stem}.png', 0) > 127
pred = predict_prob(m, img, dev) > 0.5
ov = img.copy()
for mask, col in ((gt, (0,255,0)), (pred, (0,0,255))):
    cs, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    cv2.drawContours(ov, cs, -1, col, 1)
plt.figure(figsize=(9, 8)); plt.imshow(ov[:, :, ::-1]); plt.title(f'{stem}: green = label, red = U-Net'); plt.axis('off')

## 6. Download the trained model

Put `unet_best.pt` in `WaterDropML/models/` on your laptop, then run:
```
python src/unet/predict.py
```

In [ ]:
files.download('models/unet_best.pt')
files.download('models/train_log.csv')